# A Simple Machine Learning Workflow 

This notebook will guide you through a basic supervised machine learning workflow.

> 💡 If you have brought your own dataset, try applying these steps to it at the end of this notebook. 

<img src="images/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 150px;">
<img src="images/02_enriching.png" alt="Enriching diagram" style="max-width: 150px;">
<img src="images/03_vectorization.png" alt="Vectorization diagram" style="max-width: 150px;">
<img src="images/05_modelling.png" alt="Modelling diagram" style="max-width: 150px;">
<img src="images/06_evaluation.png" alt="Evaluation diagram" style="max-width: 150px;">


We will use the EvalTweet data you downloaded in your preparation for this course to work with either of the following datasets: 



Your **goal** is to train a basic supervised machine learning model to classify text for this dataset: 
- the objective is to **predict whether a tweet (X comment) is offensive or *not* offensive**.

You will do this by training the model on the 'train' subset of the dataset, and then evaluating its performance on the 'test' subset.



# 0. Setup 

In [ ]:
#packages for loading and processing data - 
import os #for os operations
from glob import glob #for filepath operations
import pandas as pd #for dataframes
import numpy as np #for numerical operations
from datasets import load_dataset, load_from_disk               #downloads datasets from the Hugging Face Hub, and load them from disk after you have saved them


#packages required for our ML pipeline
from sklearn.preprocessing import LabelEncoder #for encoding labels as numbers
from sklearn.model_selection import train_test_split #for splitting data into training and test sets
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer #a vectorizer (converts text to numbers)
from sklearn.naive_bayes import MultinomialNB #the model (a Naive Bayes classifier)
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report #for model evaluation


#optional packages for other classifiers and pipelines:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

#I want vscode to display the full text of the tweets, so we can inspect whats happening, so I will set the max column width to None
pd.set_option('display.max_colwidth', None)

## 1. Load the data 


> 💡 As you have seen this-morning, vectorization (which comes after pre-processing) can actually take care of a lot of pre-processing for you (e.g., stopword removal, lemmatization, etc). However, there are two reasons why pre-processing remans important:
- The first is an **input reason**: your text might not be very 'clean' to begin with. It might contain non-unicode characters or html tags (e.g. <\/br>) that will confuse the vectorizer. 
- The second it an **output reason**: perhaps you want something different or targeted than a verbatim copy of the text, that is motivated by theoretical / reserach question reasons. 

⚠️ Therefore, think carefully about what pre-processing steps you want to apply, and why. You can always come back and change this part of the pipeline at any time. Do experiment. 


In [ ]:
# If you have already downloaded and save the data, you can proceed to the next step. If not, you can download the data from the following link:

tweets = load_dataset("cardiffnlp/tweet_eval", "offensive")

# Save to a local directory
tweets.save_to_disk("data/tweets_dataset")

In [ ]:
#For Colab, uncomment and run this cell to download the dataset and save it to your Google Drive. You will be prompted to authenticate your Google account.:

#from datasets import load_dataset
#from google.colab import drive
#
## Mount Google Drive
#drive.mount('/content/drive')
#
## Load dataset
#tweets = load_dataset("cardiffnlp/tweet_eval", "offensive")
#
## Save to Google Drive
#tweets.save_to_disk("/content/drive/My Drive/tweets_dataset")

In [ ]:
#first, let's define where this dataset is stored on your computer:
datadir = "data/" # or '/Users/rupertkiddle/Downloads/' -> wherever you have saved the dataset on your computer

#load the data into a df using .csv: 

tweets = load_from_disk(os.path.join(datadir,"tweets_dataset"))



In [ ]:
#convert data to pandas dataframe for easier manipulation
train_tweets = tweets["train"].to_pandas()
test_tweets = tweets["test"].to_pandas()

print(train_tweets.shape) #check the shape of the dataframe after removing missing values, it shows the number of rows and columns in the dataframe
train_tweets.head()

In [ ]:
#MISSING DATA - 
#i.e., check if any rows have missing data, and remove them if so.
print(train_tweets.isnull().sum()) #check how many missing values there are in each column

#if some rows have missing data, we can remove them using the dropna() function:
#train_tweets = train_tweets.dropna() #remove rows with missing data

# 2. Preprocessing, Tokenization and Vectorization

In this step, we will clean our text, chop it up into meaningful pieces and transform them into a numerical format that can be used by machine learning algorithms. 
We have practiced with this already on day 1, so now we will skim over these steps fast, but note that there's much room for improvement of this classifier using the tricks of day 1!   

<img src="images/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 150px;">

<img src="images/03_vectorization.png" alt="Vectorization diagram" style="max-width: 150px;">



In [ ]:
#COUNT VECTORIZER
#Remember from Monday that we could also opt for TfidfVectorizer

#Let's define a CountVectorizer with some parameters (you can experiment with these):
vectorizer_CV = CountVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.8)

#fit the vectorizer to the enriched text, and transform the text to a document-term matrix
X_CV = vectorizer_CV.fit_transform(train_tweets['text'])

#print the number of terms (i.e., columns) in the document-term matrix
print(f"CountVectorizer - number of features: {len(vectorizer_CV.get_feature_names_out())}")

#print the number of documents (i.e., rows) in the document-term matrix:
print(f"CountVectorizer - number of documents: {X_CV.shape[0]}")

# 3. Modelling
<img src="images/05_modelling.png" alt="Modelling diagram" style="max-width: 150px;">

In this step, we will train a simple supervised machine learning model (Multinomial Naive Bayes) on the 'training' data subset. 

Remember, we refer to to: 
- The 'features' as **X** (the input data, i.e., the vectorized text data)
- The 'labels' as **y** (the output data, i.e., the sentiment labels for the movie reviews, or the outlet labels for the news articles)

In [ ]:
#We need to encode the labels (i.e., the news outlets) as numbers: 
le = LabelEncoder()
y = le.fit_transform(train_tweets['label']) #encode the labels as numbers

#print what outlets they correspond to:
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
#Since we already have a separate test set, we don't need to split the data into training and test sets. Instead, we can use the training set for training and the test set for evaluation.
X_train = X_CV
y_train = train_tweets['label']

X_test = vectorizer_CV.transform(test_tweets['text'])
y_test = le.transform(test_tweets['label'])


In [ ]:
#Now, we everything is ready to train a model!

#We will use a simple Multinomial Naive Bayes classifier:
model = MultinomialNB() #here we instantiate (create) the model so that we can use it. 

#fit the model to the training data:
model.fit(X_train, y_train) #giving it the training data (X_train) and the labels (y_train).DS_Store

#predict the labels for the test data:
#NOTE: we need the true labels (y_test) out to evaluate the model in the next step.
y_pred = model.predict(X_test) #predict the labels for the test data (X_test)

# 4. Evaluation
<img src="images/06_evaluation.png" alt="Evaluation diagram" style="max-width: 150px;">

In this step, we will evaluate the performance of our trained model, b

In [ ]:
#we can also visualize the confusion matrix

print(pd.DataFrame(confusion_matrix(y_test, y_pred), index=le.classes_, columns=le.classes_))

#or we can visualize it as a dataframe with the labels as the index and columns:
label_names = tweets["test"].features["label"].names
cm = confusion_matrix(
    y_test, 
    y_pred, 
    labels=list(range(len(label_names)))
)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=label_names,
    columns=label_names
)
print(confusion_matrix_df)

In [ ]:
#the classification report gives us precision, recall, f1-score for each class (i.e., each label)
print(classification_report(y_test, y_pred, target_names=tweets["test"].features["label"].names))

#for now, higher = better (we will cover these metrics on D4).DS_Store
#if curious: 
#Precision = TP / (TP + FP) — of predicted positives, how many are correct?
#Recall = TP / (TP + FN) — of actual positives, how many did you find?
#F1 = harmonic mean of precision and recall.
#support = number of occurances of the class (the label).

> 💡 How did it perform? Better, or worse, for certain classes (labels)? Consider returning to your pre-processing and vectorization steps, to see the effects of different choices.

# 5. "All Together Now" (optional)
<img src="images/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 120px;">
<img src="images/03_vectorization.png" alt="Vectorization diagram" style="max-width: 120px;">
<img src="images/05_modelling.png" alt="Modelling diagram" style="max-width: 120px;">
<img src="images/06_evaluation.png" alt="Evaluation diagram" style="max-width: 120px;">

Sklearn's Pipeline() allows us to combine multiple steps into a single object. This is very useful for automating the process of building and evaluating models, especially when we want to systematically test different configurations (e.g., different pre-processing steps, vectorizers, models, etc). 

In [ ]:
#import it like this: 
from sklearn.pipeline import Pipeline

#we can then create a pipeline with our vectorizer and model:
#NOTE: mouse over the Pipeline() function to see what else you can add.
my_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.8)),
    ('model', MultinomialNB())
])

# Load the appropriate data sets again, note that we are using the raw text data (train_tweets['text'] and test_tweets['text']) instead of the document-term matrix (X_CV) that we created earlier. 
# This is because the pipeline will handle the vectorization for us.:
X_train = train_tweets['text']
y_train = train_tweets['label']

X_test = test_tweets['text']
y_test = test_tweets['label']

#fit the pipeline to the training data:
my_pipeline.fit(X_train, y_train)

#predict the labels for the test data:
y_pred = my_pipeline.predict(X_test)

> 💡 That's it! Pipeline just allows us to streamline the process of building and evaluating our model by encapsulating all the steps into a single object. This becomes essential for automatically testing different configurations to see what performs best - a 'grid search' - which we will explore on Day 4. 

# 6. Compare different vectorizers and classifiers (optional)

In [ ]:
configurations = [('NB with Count', CountVectorizer(min_df=5, max_df=.5), MultinomialNB()),
                 ('NB with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), MultinomialNB()),
                 ('LogReg with Count', CountVectorizer(min_df=5, max_df=.5), LogisticRegression(solver='liblinear')),
                 ('LogReg with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), LogisticRegression(solver='liblinear')),
                 ('SVM with Count - linear kernel', CountVectorizer(min_df=5, max_df=.5), SVC(kernel='linear')),
                 ('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=.5), SVC(kernel='linear')),
                 ('Random Forest with Count', CountVectorizer(min_df=5, max_df=.5), RandomForestClassifier(n_estimators=100, random_state=42)),
                 ('Random Forest with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), RandomForestClassifier(n_estimators=100, random_state=42)),
                 ]

for description, vectorizer, classifier in configurations:
    print(description)
    X_tr = vectorizer.fit_transform(X_train)
    X_te = vectorizer.transform(X_test)
    classifier.fit(X_tr, y_train)
    y_pred = classifier.predict(X_te)
    
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {f1_macro:.4f}")
    print('\n')

# 7. Try on your own data (optional)

In [ ]:
#...